# Lab 11 — Cortex Analyst: Text-to-SQL

Cortex Analyst converts natural language questions to SQL queries using a **semantic model**.

| Item | Detail |
|---|---|
| **Feature** | Cortex Analyst, Semantic Models, Semantic Views |
| **Data** | TPC-H sales views + YAML semantic model |
| **Exam Domain** | 2.0 Gen AI & LLM Functions |
| **You'll learn** | Semantic models, verified queries (VQR), suggested questions, semantic views |


## Step 1 — Environment & Data Setup

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;

-- Database pre-created by SYSADMIN in 00-admin-setup
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

In [ ]:
CREATE OR REPLACE VIEW sales_orders AS
SELECT o_orderkey AS order_key, o_custkey AS customer_key,
    o_orderstatus AS order_status, o_totalprice AS total_price,
    o_orderdate AS order_date, o_orderpriority AS order_priority
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.ORDERS;

CREATE OR REPLACE VIEW sales_lineitems AS
SELECT l_orderkey AS order_key, l_partkey AS part_key,
    l_quantity AS quantity, l_extendedprice AS extended_price,
    l_discount AS discount, l_shipdate AS ship_date, l_shipmode AS ship_mode
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.LINEITEM;

CREATE OR REPLACE VIEW sales_customers AS
SELECT c_custkey AS customer_key, c_name AS customer_name,
    c_nationkey AS nation_key, c_mktsegment AS market_segment
FROM SNOWFLAKE_SAMPLE_DATA.TPCH_SF1.CUSTOMER;

## Step 2 — The Semantic Model (YAML)

A semantic model describes your data in business terms. It contains:
- **Tables**: The underlying views/tables with column descriptions
- **Verified Queries (VQR)**: Known-good question→SQL pairs that improve accuracy
- **Relationships**: How tables join together

See `sales_semantic_model.yaml` in this directory for the full model.


In [ ]:
CREATE STAGE IF NOT EXISTS semantic_models;

-- Upload the YAML file to the stage
PUT 'file://04-analyst-and-agents/11-cortex-analyst/sales_semantic_model.yaml' @semantic_models AUTO_COMPRESS=FALSE OVERWRITE=TRUE;

## Step 3 — Verified Queries (VQR)

VQRs are the **most important accuracy lever** for Cortex Analyst.
They provide example question→SQL pairs that guide the text-to-SQL generation.

Our semantic model includes VQRs like:
- 'What is the total revenue?' → `SELECT SUM(extended_price) ...`
- 'Show revenue by market segment' → `SELECT market_segment, SUM(...) ...`


## Step 4 — Suggested Questions & custom_instructions

**Suggested questions**: Pre-built prompts shown to users for quick starts.

**custom_instructions**: Global prompt that guides all Analyst responses, e.g.:
```yaml
custom_instructions: |
  Always include ORDER BY in queries. Format currency with 2 decimal places.
  When asked about 'revenue', use extended_price column.
```


## Step 5 — Semantic Views

A **semantic view** is a database object that wraps a semantic model YAML.
Benefits over stage files:
- Version controlled in Snowflake
- Can be granted to roles (RBAC)
- Referenced by fully-qualified name


In [ ]:
CREATE OR REPLACE SEMANTIC VIEW sales_analyst_view
    COMMENT = 'Semantic view for TPC-H sales analytics'
    AS SEMANTIC MODEL FROM @semantic_models/sales_semantic_model.yaml;

## Key Takeaways

- **Semantic model** = YAML that describes tables, columns, relationships in business terms.
- **Verified Queries (VQR)** are the #1 accuracy lever — add common question→SQL pairs.
- **Suggested questions** improve user experience with pre-built prompts.
- **custom_instructions** guide all Analyst responses globally.
- **Semantic views** wrap YAML into a database object with RBAC support.
- Cortex Analyst is accessed via REST API or Streamlit — not direct SQL.
